# SEBAL Notebook 1: SEBAL Landsat Data Acquisition Workflow
### Study Area: Mississippi Delta  
### Date: 2020-01-19

This notebook performs automated acquisition of Landsat Collection-2 Level-2 data required for SEBAL evapotranspiration analysis.

Workflow steps:

1. Import required libraries and utility functions  
2. Define project directory structure  
3. Configure user inputs (study area and date)  
4. Search Landsat scenes using STAC catalog  
5. Download required Landsat bands for SEBAL processing

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import pandas as pd
from Utility import getscenes
from Utility import get_link
from Utility import download_scene

## 2. Define SEBAL Project Directory Structure

This section defines the folder structure used throughout the workflow:

- `01_raw_landsat` — downloaded Landsat imagery  
- `02_raw_aorc` — raw meteorological forcing data  
- `03_processed_met` — processed meteorology  
- `04_indices` — vegetation and surface indices  
- `05_visuals` — Visual plots

In [2]:
# get the SEBAL project root (one level above the scripts folder)
project_root = os.path.dirname(os.getcwd())
# define raw Landsat data folder path
raw_landsat = os.path.join(project_root, "01_raw_landsat")

## 3. Set Working Directory

Change the working directory to the SEBAL project root to ensure all files are saved in the correct project folders.

In [3]:
# change working directory to project root for consistency
os.chdir(project_root)

## 4. Define User Inputs

Specify the study region, Landsat dataset, and acquisition date.

Parameters:
- **bbox** — geographic bounding box of study area  
- **scenes** — Landsat collection to query  
- **date** — satellite acquisition date

In [4]:
# --- user inputs (edit here) ---

# bounding box for Mississippi Delta region (lon_min, lat_min, lon_max, lat_max)
bbox = [-91.5, 33.0, -89.5, 35.5]

# STAC collection name for Landsat Collection 2 Level 2 products
scenes = ["landsat-c2-l2"]

# date of Landsat acquisition (YYYY-MM-DD)
date = "2020-01-19"

## 5. Search Landsat Scenes

Query the Planetary Computer STAC catalog to identify Landsat scenes that intersect the study area on the selected date.

In [5]:
# --- search Landsat scenes for this date and bounding box ---
items = getscenes(bbox, scenes, date)

Number of set for the given box on 2020-01-19: 3
Scene scene ID of the first Landsat image LE07_L2SP_023037_20200119_02_T1
List of all scenes:
LE07_L2SP_023037_20200119_02_T1
LE07_L2SP_023036_20200119_02_T1
LE07_L2SP_023035_20200119_02_T1


## 6. Download Landsat Bands

For each Landsat scene found, the required SEBAL bands are downloaded and saved to the `01_raw_landsat` folder.

In [6]:
# --- Landsat bands required for SEBAL processing ---

bands = [
    "blue",      # surface reflectance band 1
    "green",     # surface reflectance band 2
    "red",       # used for NDVI
    "nir08",     # used for NDVI
    "swir16",    # used for albedo
    "swir22",    # used for albedo
    "lwir",      # thermal band for LST
    "qa_pixel"   # quality mask (cloud detection)
]

## Workflow Output

Downloaded files are stored in:

`01_raw_landsat`

These files will be used in subsequent steps for:

- NDVI computation
- surface albedo estimation
- land surface temperature (LST)
- SEBAL energy balance calculations

In [7]:
# --- download all required SEBAL bands for each Landsat scene 
records = []

for item in items:

    print(f"\nProcessing scene: {item.id}")

    for band in bands:

        print(f"Downloading band: {band}")

        band_url = get_link(item, band_name=band)

        file_path = download_scene(band_url, dir2save=raw_landsat)

        records.append({
            "scene_id": item.id,
            "band": band,
            "file": file_path
        })
df = pd.DataFrame(records)

df.to_csv("landsat_downloaded_files.csv", index=False)


Processing scene: LE07_L2SP_023037_20200119_02_T1
https://landsateuwest.blob.core.windows.net/landsat-c2/level-2/standard/etm/2020/023/037/LE07_L2SP_023037_20200119_20200822_02_T1/LE07_L2SP_023037_20200119_20200822_02_T1_SR_B1.TIF?st=2026-03-06T14%3A12%3A39Z&se=2026-03-07T14%3A57%3A39Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-03-07T11%3A04%3A28Z&ske=2026-03-14T11%3A04%3A28Z&sks=b&skv=2025-07-05&sig=kikFOvlDIuSNG/UQU/ppgWcz7k/u0/LL83gOIiNGCio%3D
Extracted Landsat band file name: LE07_L2SP_023037_20200119_20200822_02_T1_SR_B1.TIF
Downloaded as G:\Collaborations\Mentee\UF_Anitha Madapakula\Scripts\Python\SEBAL_20200119_MSDelta\01_raw_landsat\LE07_L2SP_023037_20200119_20200822_02_T1_SR_B1.TIF
https://landsateuwest.blob.core.windows.net/landsat-c2/level-2/standard/etm/2020/023/037/LE07_L2SP_023037_20200119_20200822_02_T1/LE07_L2SP_023037_20200119_20200822_02_T1_SR_B2.TIF?st=2026-03-06T14%3A12%3A39Z&se=2026-03-07